# 03 Spielorte, Laender und Geocodes vorbereiten

Dieses Notebook implementiert BD-09 und BD-10. Es liest die in BD-05 erzeugten StatsBomb-Matchdaten, extrahiert eindeutige Spielorte und bereitet daraus eine stabile Venue-Tabelle fuer Wetterabfragen vor.

BD-09 erzeugt die eindeutigen Spielorte. BD-10 fuehrt anschliessend Geocoding auf Stadt-/Land-Ebene aus, nutzt einen JSON-Cache und dokumentiert fehlende oder unsichere Lookups.

## Methodischer Ansatz

Die Venue-Daten werden zunaechst tabellarisch normalisiert: relevante Spalten werden ausgewaehlt, Stadion-/Stadt-/Land-Keys gebildet und Duplikate entfernt.

Der Geocoding-Schritt nutzt einen REST-Endpunkt und einen stabilen JSON-Cache. Dadurch werden wiederholte API-Abfragen vermieden und die getroffenen Lookup-Entscheidungen bleiben nachvollziehbar.

In [ ]:
from datetime import datetime, timezone
import json
import re
import time
import unicodedata

import pandas as pd
import requests

from project_paths import project_root as get_project_root
from pipeline_config import strip_accents

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 30)

base_path = get_project_root()
docs_path = base_path / 'docs'
bronze_path = base_path / 'data' / 'bronze'
silver_path = base_path / 'data' / 'silver'
cache_path = base_path / 'data' / 'cache'
outputs_tables_path = base_path / 'outputs' / 'tables'

matches_path = bronze_path / 'statsbomb_matches.parquet'
venues_output_path = bronze_path / 'venues_unique.parquet'
problem_cases_output_path = outputs_tables_path / 'bd09_venue_problem_cases.csv'
geocoding_cache_path = cache_path / 'geocoding_cache.json'
geocodes_output_path = silver_path / 'venue_geocodes.parquet'
geocoding_failures_output_path = outputs_tables_path / 'geocoding_failures.csv'
city_overrides_path = base_path / 'data' / 'reference' / 'venue_city_overrides.csv'

for path in [bronze_path, silver_path, cache_path, outputs_tables_path]:
    path.mkdir(parents=True, exist_ok=True)

{
    'base_path': str(base_path),
    'matches_input': 'data/bronze/statsbomb_matches.parquet',
    'city_overrides_input': 'data/reference/venue_city_overrides.csv',
    'venues_output': 'data/bronze/venues_unique.parquet',
    'problem_cases_output': 'outputs/tables/bd09_venue_problem_cases.csv',
    'geocodes_output': 'data/silver/venue_geocodes.parquet',
    'geocoding_cache_output': 'data/cache/geocoding_cache.json',
    'geocoding_failures_output': 'outputs/tables/geocoding_failures.csv',
}


## Bronze-Matchdaten lesen

Die Venue-Information kommt direkt aus der StatsBomb-Matchtabelle. StatsBomb liefert Stadion und Land, aber keine eigene City-Spalte. Die City wird deshalb im naechsten Schritt explizit normalisiert.

In [ ]:
matches = pd.read_parquet(matches_path)

required_columns = [
    'match_id',
    'match_date',
    'short_name',
    'competition_name',
    'season_name',
    'stadium_id',
    'stadium_name',
    'stadium_country_name',
]
missing_columns = sorted(set(required_columns) - set(matches.columns))
assert not missing_columns, f'Missing columns in StatsBomb matches: {missing_columns}'

venue_source = matches[required_columns].copy()

{
    'match_rows': len(matches),
    'unique_matches': int(matches['match_id'].nunique()),
    'unique_stadium_country_pairs': int(matches[['stadium_name', 'stadium_country_name']].drop_duplicates().shape[0]),
    'missing_stadium_names': int(matches['stadium_name'].isna().sum()),
    'missing_stadium_countries': int(matches['stadium_country_name'].isna().sum()),
}

## City- und Venue-Keys bilden

`location_key` identifiziert Stadt und Land. `venue_key` nutzt zusaetzlich das Stadion, damit mehrere Stadien in derselben Stadt nicht zusammenfallen. Damit kann BD-10 entscheiden, ob die Wetterabfrage auf Stadt- oder Stadionebene gecacht wird.

In [ ]:
city_overrides = pd.read_csv(city_overrides_path)
required_override_columns = {'stadium_lookup_name', 'city_name'}
missing_override_columns = required_override_columns - set(city_overrides.columns)
assert not missing_override_columns, f'Missing override columns: {sorted(missing_override_columns)}'
assert city_overrides['stadium_lookup_name'].is_unique, 'Venue city overrides must be unique by stadium lookup name.'

CITY_OVERRIDES = dict(zip(city_overrides['stadium_lookup_name'], city_overrides['city_name']))

def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).replace("''", "'").strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def lookup_text(value):
    if pd.isna(value):
        return pd.NA
    return strip_accents(clean_text(value))


def make_key(*parts):
    cleaned_parts = []
    for part in parts:
        if pd.isna(part):
            return pd.NA
        normalized = strip_accents(str(part)).lower()
        normalized = re.sub(r'[^a-z0-9]+', '_', normalized).strip('_')
        cleaned_parts.append(normalized)
    return '__'.join(cleaned_parts)


venue_base = (
    venue_source
    .assign(
        stadium_name_clean=lambda df: df['stadium_name'].map(clean_text),
        stadium_lookup_name=lambda df: df['stadium_name'].map(lookup_text),
        country_name=lambda df: df['stadium_country_name'].map(clean_text),
        match_date=lambda df: pd.to_datetime(df['match_date']),
    )
)

venue_base['city_name'] = venue_base['stadium_lookup_name'].map(CITY_OVERRIDES)
venue_base['location_key'] = venue_base.apply(lambda row: make_key(row['city_name'], row['country_name']), axis=1)
venue_base['venue_key'] = venue_base.apply(lambda row: make_key(row['city_name'], row['country_name'], row['stadium_name_clean']), axis=1)
venue_base['geocoding_query'] = venue_base.apply(
    lambda row: pd.NA if pd.isna(row['city_name']) or pd.isna(row['country_name']) else f"{row['stadium_name_clean']}, {row['city_name']}, {row['country_name']}",
    axis=1,
)

venues_unique = (
    venue_base
    .groupby(['venue_key', 'location_key', 'stadium_id', 'stadium_name', 'stadium_name_clean', 'city_name', 'country_name', 'geocoding_query'], dropna=False)
    .agg(
        match_count=('match_id', 'nunique'),
        first_match_date=('match_date', 'min'),
        last_match_date=('match_date', 'max'),
        tournaments=('short_name', lambda values: ', '.join(sorted(values.dropna().unique()))),
    )
    .reset_index()
    .sort_values(['country_name', 'city_name', 'stadium_name_clean'])
    .reset_index(drop=True)
)

venues_unique.head(20)


## Problemfaelle dokumentieren

Die Tabelle sammelt fehlende Pflichtfelder, mehrdeutige City-Keys und auffaellige Stadionnamen. Diese Faelle blockieren BD-09 nicht automatisch, sind aber fuer BD-10 sichtbar.

In [ ]:
problem_rows = []

for _, row in venues_unique.iterrows():
    missing_fields = [
        column for column in ['stadium_name', 'city_name', 'country_name', 'venue_key', 'location_key']
        if pd.isna(row[column])
    ]
    if missing_fields:
        problem_rows.append({
            'problem_type': 'missing_required_location_field',
            'severity': 'error',
            'venue_key': row['venue_key'],
            'location_key': row['location_key'],
            'stadium_name': row['stadium_name'],
            'city_name': row['city_name'],
            'country_name': row['country_name'],
            'details': ', '.join(missing_fields),
        })

duplicate_venue_keys = venues_unique[venues_unique['venue_key'].duplicated(keep=False)]
for _, row in duplicate_venue_keys.iterrows():
    problem_rows.append({
        'problem_type': 'duplicate_venue_key',
        'severity': 'error',
        'venue_key': row['venue_key'],
        'location_key': row['location_key'],
        'stadium_name': row['stadium_name'],
        'city_name': row['city_name'],
        'country_name': row['country_name'],
        'details': 'More than one output row has the same venue_key.',
    })

multi_stadium_locations = (
    venues_unique
    .groupby('location_key', dropna=False)
    .agg(stadium_count=('stadium_name', 'nunique'))
    .query('stadium_count > 1')
    .reset_index()
)
multi_stadium_keys = set(multi_stadium_locations['location_key'].dropna())
for _, row in venues_unique[venues_unique['location_key'].isin(multi_stadium_keys)].iterrows():
    problem_rows.append({
        'problem_type': 'location_key_has_multiple_stadiums',
        'severity': 'info',
        'venue_key': row['venue_key'],
        'location_key': row['location_key'],
        'stadium_name': row['stadium_name'],
        'city_name': row['city_name'],
        'country_name': row['country_name'],
        'details': 'City/country is shared by multiple stadiums; keep stadium in venue_key for disambiguation.',
    })

training_ground_rows = venues_unique[venues_unique['stadium_name_clean'].str.contains('Trainingszentrum', na=False)]
for _, row in training_ground_rows.iterrows():
    problem_rows.append({
        'problem_type': 'venue_name_looks_like_training_ground',
        'severity': 'warning',
        'venue_key': row['venue_key'],
        'location_key': row['location_key'],
        'stadium_name': row['stadium_name'],
        'city_name': row['city_name'],
        'country_name': row['country_name'],
        'details': 'StatsBomb venue name should be reviewed before BD-10 geocoding.',
    })

problem_cases = pd.DataFrame(problem_rows, columns=[
    'problem_type',
    'severity',
    'venue_key',
    'location_key',
    'stadium_name',
    'city_name',
    'country_name',
    'details',
])

problem_cases.to_csv(problem_cases_output_path, index=False)

{
    'problem_cases': len(problem_cases),
    'problem_types': problem_cases['problem_type'].value_counts().to_dict() if not problem_cases.empty else {},
    'problem_cases_output': 'outputs/tables/bd09_venue_problem_cases.csv',
}

## Output speichern

In [ ]:
output_columns = [
    'venue_key',
    'location_key',
    'stadium_id',
    'stadium_name',
    'stadium_name_clean',
    'city_name',
    'country_name',
    'geocoding_query',
    'match_count',
    'first_match_date',
    'last_match_date',
    'tournaments',
]

venues_unique = venues_unique[output_columns]
venues_unique.to_parquet(venues_output_path, index=False)

{
    'venue_rows': len(venues_unique),
    'unique_venue_keys': int(venues_unique['venue_key'].nunique(dropna=True)),
    'unique_location_keys': int(venues_unique['location_key'].nunique(dropna=True)),
    'total_match_references': int(venues_unique['match_count'].sum()),
    'venues_output': 'data/bronze/venues_unique.parquet',
}

## Qualitaetschecks

Die Checks bilden die BD-09-Akzeptanzkriterien ab: Alle eindeutigen Spielorte sind vorhanden, gleiche Stadion-/Land-Kombinationen sind dedupliziert und Problemfaelle liegen als Tabelle vor.

In [ ]:
expected_venue_pairs = venue_source[['stadium_name', 'stadium_country_name']].drop_duplicates().shape[0]
missing_required_values = venues_unique[['venue_key', 'location_key', 'stadium_name', 'city_name', 'country_name']].isna().sum().to_dict()
duplicate_venue_key_rows = int(venues_unique['venue_key'].duplicated(keep=False).sum())

quality_checks_bd09 = {
    'input_stadium_country_pairs': int(expected_venue_pairs),
    'output_venue_rows': len(venues_unique),
    'output_exists': venues_output_path.exists(),
    'problem_table_exists': problem_cases_output_path.exists(),
    'missing_required_values': missing_required_values,
    'duplicate_venue_key_rows': duplicate_venue_key_rows,
    'problem_cases': len(problem_cases),
}

assert quality_checks_bd09['output_exists']
assert quality_checks_bd09['problem_table_exists']
assert quality_checks_bd09['output_venue_rows'] == quality_checks_bd09['input_stadium_country_pairs']
assert all(value == 0 for value in missing_required_values.values())
assert duplicate_venue_key_rows == 0

quality_checks_bd09

In [ ]:
venues_unique.head(20)

## Zwischenergebnis BD-09

BD-09 ist erfuellt, wenn die Assertions erfolgreich sind und `data/bronze/venues_unique.parquet` sowie `outputs/tables/bd09_venue_problem_cases.csv` existieren. Die Venue-Tabelle ist fuer BD-10 vorbereitet: Sie enthaelt eindeutige Stadion-Keys, City-/Country-Keys und eine konkrete `geocoding_query` pro Spielort.